# ЛР1 · 02. Текстовая LLM как модель прогноза: `microsoft/Phi-3.5-mini-instruct`

Подход LLMTime (Gruver et al., NeurIPS 2023): ряд записывается текстом, модель продолжает его.
Протокол — `PROTOCOL.md`, п. 4–5. Ответ строго JSON `{"forecast": [24 целых числа]}`, `temperature = 0` (жадная генерация).

Порядок прогонов (по важности — при сбое главное уже сохранено):
1. отладка на val, промпт P2;
2. тест, P2 — основной zero-shot;
3. тест, P1 и P3 — чувствительность к формулировке;
4. тест, few-shot: P2 + 3 примера «неделя → сутки» той же станции из train;
5. тест, P2 с контекстом 168 и 672 ч — чувствительность к длине контекста.

**Для Phi-3.5 выполняются только пункты 1–2** (из-за времени, см. `PROTOCOL.md`, п. 9).

Сохраняются метрики (`runs.csv`), прогнозы и **сырые ответы модели** (`responses__*.parquet`) — для анализа невалидных ответов.

**Kaggle:** Inputs — датасет и вывод `lab-00-setup`; Accelerator — **GPU T4 x2**; Internet — **On**.

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers>=4.45", "accelerate"],
                   capture_output=True, text=True)
print("pip:", "OK" if r.returncode == 0 else r.stderr[-1000:])

In [ ]:
import glob, json, re, os, sys, time, traceback, gc, importlib.metadata
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"     # меньше фрагментации памяти GPU
from pathlib import Path
import numpy as np, pandas as pd, torch

hits = glob.glob("/kaggle/input/**/labkit.py", recursive=True) or glob.glob("labkit.py")
LAB00 = Path(hits[0]).parent
sys.path.insert(0, str(LAB00))
import labkit as lk

MODEL_ID = "microsoft/Phi-3.5-mini-instruct"
SHORT = "phi35_mini"
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "нет")
print("transformers", importlib.metadata.version("transformers"))

## 1. Данные и окна

In [ ]:
d = lk.load()
sub = json.load(open(LAB00 / "subset.json", encoding="utf-8"))
stations = sub["stations"]
test_orig = [pd.Timestamp(x) for x in sub["test_origins"]]
val_orig = [pd.Timestamp(x) for x in sub["val_origins"]]
assert stations == lk.make_subset(d)
tmax = d.train_max()
W = {("val", 336): lk.make_windows(d, stations, val_orig, context=336)}
for c in lk.CONTEXTS:
    W[("test", c)] = lk.make_windows(d, stations, test_orig, context=c)

# Few-shot: 3 примера той же станции из train — последние 3 одноимённых дня недели до конца train,
# «168 ч истории → следующие 24 ч». Утечки нет: всё до 31.03.2026.
idx = d.y.index
train_end = d.part("train")[1]
def fewshot_examples(station, origin, k=3, ctx=168):
    out, day = [], (train_end + pd.Timedelta(hours=1)).normalize() - pd.Timedelta(days=1)
    while len(out) < k:
        if day.dayofweek == origin.dayofweek and day + pd.Timedelta(hours=23) <= train_end:
            p = idx.get_loc(day)
            hist = d.y[station].iloc[p - ctx:p].to_numpy()
            fut = d.y[station].iloc[p:p + lk.HORIZON].to_numpy()
            closed = d.closed[station].iloc[p - ctx:p + lk.HORIZON].any()
            if not closed:
                out.append((day, hist, fut))
        day -= pd.Timedelta(days=1)
    return out[::-1]

## 2. Промпты

In [ ]:
DAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
SYSTEM = "You are a time series forecasting model. You answer only with JSON."
FORMAT = ('Return only JSON in the form {"forecast": [24 integers]}: exactly 24 numbers, one per hour '
          'from 00:00 to 23:00 of the forecast day, with no other text.')

def series_text(x):
    return ", ".join(str(int(round(v))) for v in x)

def prompt_p1(ctx, origin):
    return f"Continue the sequence with the next 24 values.\n{FORMAT}\nSequence: {series_text(ctx)}"

def describe(ctx, origin):
    start = origin - pd.Timedelta(hours=len(ctx))
    return (f"Hourly number of entries at a New York City subway station. The series below has {len(ctx)} values, "
            f"one per hour, starting {DAYS[start.dayofweek]} 00:00. "
            f"Forecast the next 24 hours: {DAYS[origin.dayofweek]} 00:00 to 23:00.")

def prompt_p2(ctx, origin):
    return f"{describe(ctx, origin)}\n{FORMAT}\nSeries: {series_text(ctx)}"

def prompt_p3(ctx, origin):
    last_week = ctx[-168:-144] if len(ctx) >= 168 else ctx[-24:]
    stats = (f"Input statistics: min {int(ctx.min())}, max {int(ctx.max())}, mean {ctx.mean():.0f}. "
             f"The series has strong daily (24 h) and weekly (168 h) seasonality. "
             f"Values on the same weekday one week ago: {series_text(last_week)}.")
    return f"{describe(ctx, origin)}\n{stats}\n{FORMAT}\nSeries: {series_text(ctx)}"

def prompt_fewshot(ctx, origin, station):
    parts = [describe(ctx, origin), "Examples from the same station (one week of history, then the next day):"]
    for day, h, f in fewshot_examples(station, origin):
        parts.append(f"History: {series_text(h)}\nAnswer: {json.dumps({'forecast': [int(v) for v in f]})}")
    parts += [FORMAT, f"Series: {series_text(ctx)}"]
    return "\n".join(parts)

PROMPTS = {"P1": prompt_p1, "P2": prompt_p2, "P3": prompt_p3}
w0 = W[("test", 336)]
example = {k: f(w0.ctx[0], w0.meta["origin"].iat[0]) for k, f in PROMPTS.items()}
example["fewshot"] = prompt_fewshot(w0.ctx[0], w0.meta["origin"].iat[0], w0.meta["station"].iat[0])
json.dump({"system": SYSTEM, **{k: v[:3000] for k, v in example.items()}},
          open(lk.results_dir() / "prompts_example.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
for k, v in example.items():
    print(f"--- {k} ({len(v)} символов) ---\n{v[:400]} …\n")

## 3. Разбор ответа

In [ ]:
def parse_forecast(text):
    """Все числа ответа (list) или None. Строгая проверка формата — отдельно, в strict_ok()."""
    return _parse(text)

def strict_ok(text, parsed):
    """Строгий формат: JSON {"forecast": [...]} ровно из 24 чисел."""
    try:
        v = json.loads(re.search(r"\{.*?\}", text, flags=re.S).group(0)).get("forecast")
        return isinstance(v, list) and len(v) == lk.HORIZON
    except Exception:
        return False

def _parse(text):
    """JSON {"forecast": [...]} → список чисел; запасной путь — числа после слова forecast. None, если не вышло."""
    m = re.search(r"\{.*?\}", text, flags=re.S)
    if m:
        try:
            v = json.loads(m.group(0)).get("forecast")
            if isinstance(v, list):
                return [float(x) for x in v]
        except Exception:
            pass
    m = re.search(r"forecast\W*\[([^\]]*)\]?", text, flags=re.S | re.I)
    if m:
        nums = re.findall(r"-?\d+(?:\.\d+)?", m.group(1))
        if nums:
            return [float(x) for x in nums]
    return None

for t in ['{"forecast": [1, 2, 3]}', 'Sure! {"forecast":[5,6]} done', 'forecast: [7, 8, 9', 'no numbers']:
    print(repr(t), "->", parse_forecast(t))

## 4. Модель и генерация

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
tok = AutoTokenizer.from_pretrained(MODEL_ID)
tok.padding_side = "left"
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map=DEVICE)
model.eval()
cfg = model.config
kv_per_token = 2 * cfg.num_hidden_layers * getattr(cfg, "num_key_value_heads", cfg.num_attention_heads) \
               * (cfg.hidden_size // cfg.num_attention_heads) * 2      # байт на токен в KV-кэше (fp16)
KV_BUDGET = 2.5e9                                                     # под KV-кэш на T4 (остальное — веса и активации)
TOKEN_BUDGET = 12000                                                  # промпт × батч: ограничение по активациям
MAX_NEW = 24 * 7 + 30
print(f"{MODEL_ID}: {sum(p.numel() for p in model.parameters()) / 1e9:.2f} млрд параметров, "
      f"KV {kv_per_token / 1024:.0f} КБ/токен")

def chat(user):
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

MAX_OK = [10**6]            # наибольшая пачка, прошедшая без нехватки памяти (запоминается между вызовами)

def generate(texts):
    """Генерация с защитой от нехватки памяти: при OOM пачка делится пополам."""
    try:
        return _generate(texts)
    except torch.OutOfMemoryError:
        gc.collect(); torch.cuda.empty_cache()
        if len(texts) == 1:
            raise
        h = len(texts) // 2
        MAX_OK[0] = min(MAX_OK[0], h)
        print(f"    нехватка памяти на пачке {len(texts)} → делю на {h} и {len(texts) - h}; дальше пачки ≤ {MAX_OK[0]}")
        return generate(texts[:h]) + generate(texts[h:])

def _generate(texts):
    enc = tok(texts, return_tensors="pt", padding=True).to(DEVICE)
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=MAX_NEW, do_sample=False, temperature=None, top_p=None,
                             top_k=None, pad_token_id=tok.pad_token_id)
    return tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)

In [ ]:
ERRORS = lk.results_dir() / "errors.csv"

def run(mode, w, make_prompt):
    """make_prompt(ctx, origin, station) -> текст. Сохраняет метрики, прогнозы и сырые ответы."""
    t0 = time.time()
    try:
        texts = [chat(make_prompt(w.ctx[i], w.meta["origin"].iat[i], w.meta["station"].iat[i])) for i in range(len(w))]
        n_tok = len(tok(texts[0])["input_ids"]) + MAX_NEW
        batch = int(max(1, min(16, KV_BUDGET // (kv_per_token * n_tok), TOKEN_BUDGET // n_tok)))
        torch.cuda.reset_peak_memory_stats()
        answers, per_window = [], []
        i = 0
        while i < len(texts):
            b = min(batch, MAX_OK[0])
            t = time.perf_counter()
            answers += generate(texts[i:i + b])
            torch.cuda.synchronize()
            per_window += [(time.perf_counter() - t) * 1000 / len(texts[i:i + b])] * len(texts[i:i + b])
            if i == 0:
                print(f"  {mode}: промпт ~{n_tok - MAX_NEW} токенов, батч {batch}, "
                      f"первый батч {per_window[0] / 1000:.1f} с/окно; ответ: {answers[0][:150]!r}")
            i += b
        parsed = [parse_forecast(a) for a in answers]
        strict = np.array([strict_ok(a, p) for a, p in zip(answers, parsed)])
        # PROTOCOL.md, п. 5 (уточнение): если чисел больше 24, берутся первые 24 — модель продолжила ряд
        # дальше горизонта; меньше 24, не разобралось, NaN или выброс — невалидно.
        point = [np.clip(np.array(p[:lk.HORIZON], dtype="float64"), 0, None) if p is not None and len(p) >= lk.HORIZON
                 else None for p in parsed]
        valid = lk.check_valid(point, w, tmax)
        point = lk.apply_fallback(point, valid, w)
        pd.DataFrame({"station": w.meta["station"], "origin": w.meta["origin"], "answer": answers,
                      "parsed_len": [len(p) if p is not None else -1 for p in parsed], "valid": valid,
                      "strict_format": strict}).to_parquet(
            lk.results_dir() / f"responses__{SHORT}__{mode}.parquet", index=False)
        timing = {"ms_p50": float(np.median(per_window)), "ms_p95": float(np.quantile(per_window, 0.95)),
                  "total_s": time.time() - t0}
        row = lk.save_run(SHORT, mode, w, point, valid, None, timing,
                          {"weights": MODEL_ID, "context": w.ctx.shape[1], "batch": batch, "prompt_tokens": n_tok - MAX_NEW,
                           "invalid_strict_share": float(1 - strict.mean())})
        print(f"  {SHORT:10s} {mode:22s} WAPE={row['wape']:.4f} [{row['wape_ci_lo']:.4f}; {row['wape_ci_hi']:.4f}] "
              f"невалидных={row['invalid_share']:.1%} (строго {1 - strict.mean():.1%}) p50={row['ms_p50']:.0f} мс/окно, всего {timing['total_s'] / 60:.1f} мин")
    except Exception as e:
        tb = traceback.format_exc()
        print(f"!!! {mode}: {e}\n{tb[-1500:]}")
        pd.DataFrame([{"date": pd.Timestamp.now(), "model": SHORT, "stage": mode, "error": repr(e)[:500],
                       "traceback": tb[-3000:]}]).to_csv(ERRORS, mode="a", header=not ERRORS.exists(), index=False)
    gc.collect(); torch.cuda.empty_cache()

## 5. Быстрая проверка (8 окон val на каждую конфигурацию) и прогоны

In [ ]:
p2 = lambda c, o, s: prompt_p2(c, o)
def subset(w, n=8):
    return lk.Windows(w.meta.iloc[:n].reset_index(drop=True), w.ctx[:n], w.target[:n], w.mask[:n], w.ctx_index[:n], w.scale[:n])
wv = W[("val", 336)]
smoke = {"P1": lambda c, o, s: prompt_p1(c, o), "P2": p2, "P3": lambda c, o, s: prompt_p3(c, o),
         "fewshot": lambda c, o, s: prompt_fewshot(c, o, s)}
for name, f in smoke.items():
    t = time.time()
    texts = [chat(f(wv.ctx[i], wv.meta["origin"].iat[i], wv.meta["station"].iat[i])) for i in range(4)]
    ans = generate(texts)
    ok = sum(p is not None and len(p) >= lk.HORIZON for p in map(parse_forecast, ans))
    print(f"проверка {name}: {ok}/4 ответов с ≥24 числами, {time.time() - t:.0f} с; пример: {ans[0][:120]!r}")

In [ ]:
run("zs_P2_ctx336_val", W[("val", 336)], p2)
run("zs_P2_ctx336_test", W[("test", 336)], p2)

# Phi-3.5 — только основной промпт P2: у модели нет GQA, KV-кэш в ~11 раз больше, чем у Qwen,
# и полная сетка заняла бы ~10 ч на T4. Чувствительность к промпту, few-shot и контексту — на Qwen (PROTOCOL.md, п. 9).

## 6. Сводка

In [ ]:
runs = pd.read_csv(lk.results_dir() / "runs.csv")
show = ["model", "mode", "n_windows", "invalid_share", "wape", "wape_ci_lo", "wape_ci_hi", "mase", "ms_p50", "gpu_peak_gb"]
print(runs[[c for c in show if c in runs]].round(4).to_string())
print("\nОшибки:\n" + (pd.read_csv(ERRORS)[["stage", "error"]].to_string() if ERRORS.exists() else "нет"))
for f in sorted(lk.results_dir().glob(f"responses__{SHORT}__*.parquet")):
    r = pd.read_parquet(f)
    bad = r[~r["valid"]]
    print(f"\n{f.name}: невалидных {len(bad)} из {len(r)}")
    for a in bad["answer"].head(2):
        print("   пример:", repr(a[:200]))